# NLP3 - BERT for Clinical Document Classification

In this lab, we will build document classification models using BERT  
The provided dataset consists of transcriptions and their corresponding medical specialties. Those specialities are our labels.

**Connect to a GPU runtime environment if you have access to one (Runtime -> Change runtime type).**

In [1]:
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from tqdm import tqdm

In [2]:
# we set the seed for reproducibility

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_value = 123
set_seed(seed_value)

First, load the sample documents for classification from the provided file `300-balanced-category-mtsamples.csv`.

In [3]:
data_folder = '.'
df_sampled = pd.read_csv(f'{data_folder}/300-balanced-category-mtsamples.csv')
df_sampled.head()

,description,medical_specialty,sample_name,transcription,keywords
0,"Ethmoidectomy, antrostomy with polyp removal,...",Surgery,Endoscopic Sinus Surgery,PREOPERATIVE DIAGNOSIS:\n\n1. Left chronic an...,NaN
1,Left facial cellulitis and possible odontogen...,Surgery,Odontogenic Abscess I&D,PREOPERATIVE DIAGNOSES:\n\n1. Left facial cel...,"surgery, odontogenic, facial cellulitis, incis..."
2,Laparoscopic hand-assisted left adrenalectomy...,Surgery,Adrenalectomy & Umbilical Hernia Repair,"PREOPERATIVE DIAGNOSES\n\n1. Adrenal mass, ri...","surgery, adrenalectomy, laparoscopic hand-assi..."
3,"Bronchoscopy brushings, washings and biopsies...",Surgery,Bronchoscopy Brushings,OPERATIVE PROCEDURE:\n\n Bronchoscopy brushin...,"surgery, mac, fluoroscopy, fiberoptic bronchos..."
4,Tracheostomy and thyroid isthmusectomy. Vent...,Surgery,Tracheostomy & Thyroid Isthmusectomy,PREOPERATIVE DIAGNOSES:\n\n1. Ventilator-depe...,"surgery, ventilator-dependent respiratory fail..."


**Task 1** Create a validation set and add a validation step in the training loop to store the best model (as evaluated on the validation set).

*TIP*: use Scikit's `train_test_split` to get three sets from the sampled data, i.e., train, validation and test.

In [4]:
from sklearn.model_selection import train_test_split

# set the randome state for reproducing
random_state = 9
test_size = 0.2

categories = df_sampled['medical_specialty'].unique().tolist()
texts = df_sampled['transcription'].tolist()
labels = df_sampled['medical_specialty'].apply(lambda x: categories.index(x)).tolist()
X_train_val, X_test, y_train_val, y_test = train_test_split(texts, labels, test_size=test_size, random_state=random_state)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=.1, random_state=random_state)

## 1. Loading data for training and testing

Importing BERT models for the first time may take a while as they need to be downloaded from [HuggingFace](https://huggingface.co).
We also download the relevant tokenizer. Recall, that it is used to convert the input text into sub-word units for BERT. It also handles tasks such as adding special tokens, padding, and truncating the input text to a fixed length.


In [5]:
from transformers import AutoTokenizer
# model_name = 'bert-base-uncased'
model_name = 'emilyalsentzer/Bio_ClinicalBERT'

# Load the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]


The tokenizer function tokenises the input texts and handles padding and truncation. Padding ensures that all input sequences are of the same length by adding special padding tokens. Truncation ensures that input sequences longer than the maximum length are truncated to fit the model's input size.


In [6]:
# let us first check if the max length is set for this tokenizer

max_length = tokenizer.model_max_length

if max_length > 10000:
    max_length = 512


In [7]:
def prepare_X_y(texts, labels):
    tokenized = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length
    )
    seq = torch.tensor(tokenized['input_ids'])
    mask = torch.tensor(tokenized['attention_mask'])
    y = torch.tensor(labels)
    return seq, mask, y

train_seq, train_mask, train_y = prepare_X_y(X_train, y_train)
val_seq, val_mask, val_y = prepare_X_y(X_val, y_val)
test_seq, test_mask, test_y = prepare_X_y(X_test, y_test)

Run the code below to output the original text and its tokenized version (subword tokens) using the BERT tokenizer. The tokenized output includes special tokens like `[CLS]`, which is used by BERT to represent the entire input sequence. Its embedding is typically fed into a classifier.

In [8]:
print(f"Text: {X_train[0]}\n")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(train_seq[0].tolist())}")

Text: PREOPERATIVE DIAGNOSIS: 

 Left pleural effusion, parapneumonic, loculated.

POSTOPERATIVE DIAGNOSIS: 

 Left pleural effusion, parapneumonic, loculated.

OPERATION: 

 Left chest tube placement.

IV SEDATION: 

 5 mg of Versed total given under pulse ox monitoring, 1% lidocaine local infiltration.

PROCEDURE: 

 With the patient semi recumbent and supine the left anterolateral chest was prepped and draped in the usual sterile fashion.  A 1% lidocaine was liberally infiltrated into the skin, subcutaneous tissue, deep fascia and the anterior axillary line just below the level of the nipple.  The incision was made and deepened through the different layers to reach the intercostal space.  The pleura was entered on top of the underlying rib and finger digital palpation was performed.  Multiple loculations were encountered.  Break up of loculations was performed posteriorly and a chest tube was directed posteriorly.  Only a small amount of fluid was noted to come out initially.  This 

Let's check that the sequences have been correctly tokenized and padded to the expected dimensions.

In [9]:
print(f"Training sequences shape: {train_seq.shape}")
print(f"Training attention masks shape: {train_mask.shape}")
print(f"Training labels shape: {train_y.shape}")


Training sequences shape: torch.Size([216, 512])
Training attention masks shape: torch.Size([216, 512])
Training labels shape: torch.Size([216])


## 2. Fine-tuning Prebuilt BERT

We define our hyperparameters.

In [10]:
# If you have access to a GPU, consider increasing the batch size to 32 and extending the number of epochs to 10-20
batch_size = 32
epochs = 10

# batch_size = 1
# epochs = 2

learning_rate = 1e-5

Then we create dataloaders.

In [11]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# Wrap tensors
training_data = TensorDataset(train_seq, train_mask, train_y)
validation_data = TensorDataset(val_seq, val_mask, val_y)
test_data = TensorDataset(test_seq, test_mask, test_y)

# Samplers for sampling the data during training, validation and testing
sampler = RandomSampler(training_data)
val_sampler = RandomSampler(validation_data)
test_sampler = RandomSampler(test_data)

# DataLoader for train, validation and test sets
training_data_loader = DataLoader(training_data, sampler=sampler, batch_size=batch_size)
validation_data_loader = DataLoader(validation_data, sampler=val_sampler, batch_size=batch_size)
test_data_loader = DataLoader(test_data, sampler=test_sampler)

The line below selects GPU if available.

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

The function below trains the model for one epoch. It will iterate over data batches, compute loss, and update model parameters.

In [13]:
def train_model_epoch(model, dataloader, optimizer, device):

    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(dataloader)):
      # unpack the batch and move data to the specified device
      batch_input_ids, batch_attention_masks, batch_labels = batch
      batch_input_ids = batch_input_ids.to(device)
      batch_attention_masks = batch_attention_masks.to(device)
      batch_labels = batch_labels.to(device)

      # clear previous gradients
      model.zero_grad()
      # compute the model output and loss
      outputs = model(batch_input_ids, attention_mask=batch_attention_masks, labels=batch_labels)
      loss = outputs.loss
      total_loss += loss.item()
      # compute gradients
      loss.backward()
      # update model parameters
      optimizer.step()

    avg_train_loss = total_loss / len(dataloader)
    # evaluate the model on the validation set
    epoch_result = evaluate(model, validation_data_loader)

    print('loss {0:.5f}'.format(avg_train_loss), 'validation performance, p:{precision:.3f}, r:{recall:.3f}, f1:{f1:.3f}'.format(**epoch_result['performance']))
    return epoch_result

Below is the function that will run the evaluation over the validation data.

In [14]:
def evaluate(model, eval_data_loader, eval_labels=[1]):
    y_true = []
    y_pred = []
    model.eval()
    with torch.no_grad():
        for batch in tqdm(eval_data_loader):
            X_batch, mask, y_batch = batch

            preds = model(X_batch.to(device), mask.to(device))

            # Compute softmax to get probabilities
            probs = torch.nn.functional.softmax(preds.logits.cpu(), dim=1)

            # Get the predicted labels
            predicted = np.argmax(probs, axis=1)

            # store true and predicted labels
            y_true += y_batch
            y_pred += predicted.cpu()

    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=eval_labels)
    cm = confusion_matrix(y_true, y_pred)
    return {
        "data": (y_true, y_pred),
        "performance": {
            "precision": p[0],
            "recall": r[0],
            "f1": f[0]
        },
        "confusion_matrix": cm
    }

We will now download a prebuilt version of BERT for sequence classification and train it.

**Task 2.** Below write a piece of code to update the best model based on the validation F1 score during training. *Hint*: within your training loop, after evaluating the model's performance for the current epoch, write the code to update `best_score` and `best_model` if the current epoch's F1 score is higher than `best_score`.

In [15]:
from transformers import BertForSequenceClassification
from torch.optim import AdamW

model_prebuilt = BertForSequenceClassification.from_pretrained(model_name, num_labels=len(categories)).to(device)

# Define optimizer
optimizer = AdamW(model_prebuilt.parameters(), lr=learning_rate)

best_model = None
best_score = -1

for epoch in range(epochs):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    #train model
    epoch_result = train_model_epoch(model_prebuilt, training_data_loader, optimizer, device)
    if best_score < epoch_result['performance']['f1']:
        best_score = epoch_result['performance']['f1']
        best_model = model_prebuilt.state_dict()
        print('current best score is {0:.3f}'.format(best_score))
    print('\n')

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the chec


 Epoch 1 / 10


  0%|          | 0/7 [00:00<?, ?it/s]

...

loss 0.28762 validation performance, p:0.800, r:0.923, f1:0.857



 Epoch 9 / 10


100%|██████████| 1/1 [00:00<00:00,  1.17it/s]


loss 0.24949 validation performance, p:0.800, r:0.923, f1:0.857



 Epoch 10 / 10


100%|██████████| 1/1 [00:00<00:00,  1.17it/s]

loss 0.24836 validation performance, p:0.800, r:0.923, f1:0.857




We will now use the `evaluate` function to assess our model.

In [16]:
# we ensure that the model being evaluated is the best version obtained during training
model_prebuilt.load_state_dict(best_model)

eval_result = evaluate(model_prebuilt, test_data_loader)
print(eval_result['performance'])

100%|██████████| 60/60 [00:00<00:00, 71.89it/s]

{'precision': np.float64(0.7333333333333333), 'recall': np.float64(0.9166666666666666), 'f1': np.float64(0.8148148148148148)}


## 3. Custom model with BERT embedded vector

In this part of the lab, we will build a custom BERT-based text classifier that averages subtoken-level BERT embeddings.

**Task 3**: Implement embedding averaging in the forward method of the `BERT_Text_Classifier` class. *Hint:* For this, get `hidden_states` from the BERT output. Use `torch.mean` to compute the average of the last hidden state along the sequence dimension.

In [17]:

# Define the BERT_Text_Classifier class
class BERT_Text_Classifier(nn.Module):
    def __init__(self, bert, class_num, bert_dim, hidden_dim):
        super(BERT_Text_Classifier, self).__init__()
        self.bert = bert
        self.dropout = nn.Dropout(0.1)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(bert_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, class_num)
        self.softmax = nn.LogSoftmax(dim=1)


    # Define the forward pass
    def forward(self, seq, attention_mask=None, labels=None):
        # Use pretrained BERT to read the sequence with the mask (pay attention to which tokens)
        bert_out = self.bert(seq, attention_mask=attention_mask, output_hidden_states=True)

        # TO DO
        # Get all hidden states
        hidden_states = bert_out.hidden_states

        # Compute the average of the last hidden state
        avg_hidden_state = torch.mean(hidden_states[-1], dim=1)
         # TO DO

        x = self.fc1(avg_hidden_state)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)

        logits = self.softmax(x)

        loss = None
        if labels is not None:
            loss_fn = nn.NLLLoss()
            loss = loss_fn(logits, labels)

        # Create an object to return so that we could reuse the previous training loop
        class Output:
          def __init__(self, loss, logits, hidden_states):
            self.loss = loss
            self.logits = logits
            self.hidden_states = hidden_states

        return Output(loss, logits, hidden_states)

In this section, we initialise and train our custom BERT-based text classifier.

In [18]:
from transformers import AutoModel

bert = AutoModel.from_pretrained(model_name).to(device)
# bert.config.hidden_size fetches the hidden size of the BERT model
# max_length sets the hidden dimension of the model based on the BERT maximum input length
model_custom = BERT_Text_Classifier(bert, class_num=len(categories), bert_dim=bert.config.hidden_size, hidden_dim=max_length).to(device)
optimizer = AdamW(model_custom.parameters(), lr=learning_rate)

best_model = None
best_score = -1

for epoch in range(epochs):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    #train model
    epoch_result = train_model_epoch(model_custom, training_data_loader, optimizer, device)
    if best_score < epoch_result['performance']['f1']:
        best_score = epoch_result['performance']['f1']
        best_model = model_custom.state_dict()
        print('current best score is {0:.3f}'.format(best_score))
    print('\n')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Epoch 1 / 10


100%|██████████| 1/1 [00:00<00:00,  1.15it/s]


loss 0.68331 validation performance, p:0.667, r:0.769, f1:0.714
current best score is 0.714



 Epoch 2 / 10


...

loss 0.24294 validation performance, p:0.765, r:1.000, f1:0.867
current best score is 0.867



 Epoch 9 / 10


100%|██████████| 1/1 [00:00<00:00,  1.15it/s]


loss 0.22479 validation performance, p:0.846, r:0.846, f1:0.846



 Epoch 10 / 10


100%|██████████| 1/1 [00:00<00:00,  1.15it/s]

loss 0.21019 validation performance, p:0.765, r:1.000, f1:0.867




Let's proceed and evaluate our custom model. How does its performance compare against the prebuilt version?

In [19]:
# we ensure that the model being evaluated is the best version obtained during training
model_custom.load_state_dict(best_model)

epoch_result = evaluate(model_custom, test_data_loader)
print(epoch_result['performance'])

100%|██████████| 60/60 [00:00<00:00, 74.11it/s]

{'precision': np.float64(0.696969696969697), 'recall': np.float64(0.9583333333333334), 'f1': np.float64(0.8070175438596491)}


**Task 4** Run the all the lab code using the specialised biomedical BERT model [`emilyalsentzer/Bio_ClinicalBERT`](https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT). How will this model perform?